# BuildMate Rentals — Bronze Layer
**Task 1 · Land every source exactly as it arrived**

Bronze is a *photograph, not a repair*. Every column is read as a **string**, nothing is
cleaned or recast, and we add only two lineage columns — `ingested_at` and `source_file`.
The 31 daily rental files are read in a single wildcard load.

> Why strings? If we let Spark infer types here, the text money column (`Rs. 42,180.00`)
> becomes `null` before we can *count* the loss in Silver. Bronze preserves the mess on purpose.

**Acceptance criteria for this notebook**
- `bronze_customers` 602, `bronze_depots` 6, `bronze_rentals` 976, `bronze_billing` 787
- every column stored as string; `ingested_at` + `source_file` present on every table
- the 10-June source-file check returns two files with the same row count


In [ ]:
from pyspark.sql import functions as F

RAW = "Files/buildmate_raw_data/raw"

def read_raw(path):
    # header on, inferSchema OFF -> every column lands as a string.
    # Add lineage: when the row was ingested, and which file it came from.
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(path)
            .withColumn("ingested_at", F.current_timestamp())
            .withColumn("source_file", F.input_file_name()))

In [ ]:
bronze_customers = read_raw(f"{RAW}/customer_master/customer_master_export.csv")
bronze_depots    = read_raw(f"{RAW}/depots/depot_master.csv")
bronze_rentals   = read_raw(f"{RAW}/rentals/rentals_*.csv")   # single wildcard = all 31 files
bronze_billing   = read_raw(f"{RAW}/billing/billing_export.csv")

for name, df in [("bronze_customers", bronze_customers),
                 ("bronze_depots",    bronze_depots),
                 ("bronze_rentals",   bronze_rentals),
                 ("bronze_billing",   bronze_billing)]:
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print("wrote", name)

### Evidence 1 — raw row counts (screenshot this)

In [ ]:
counts = {
    "bronze_customers": spark.table("bronze_customers").count(),  # expect 602
    "bronze_depots":    spark.table("bronze_depots").count(),     # expect 6
    "bronze_rentals":   spark.table("bronze_rentals").count(),    # expect 976 (10 Jun sent twice)
    "bronze_billing":   spark.table("bronze_billing").count(),    # expect 787
}
for k, v in counts.items():
    print(f"{k:20s} {v}")

assert counts == {"bronze_customers":602,"bronze_depots":6,"bronze_rentals":976,"bronze_billing":787}
print("\nBronze counts OK")

### Evidence 2 — every column is a string, lineage present

In [ ]:
# All source columns should be StringType; only ingested_at is a timestamp.
bronze_billing.printSchema()
assert "ingested_at" in bronze_rentals.columns and "source_file" in bronze_rentals.columns
non_lineage = [f for f in bronze_billing.schema.fields if f.name not in ("ingested_at",)]
assert all(str(f.dataType) == "StringType()" for f in non_lineage), "a source column was not read as string!"
print("All source columns are strings, lineage columns present.")

### Evidence 3 — prove the re-sent day
The rental system re-sent 10 June after a failed transfer. Using **lineage only**, we group the
10-June rows by their source file and show two files carrying the same rentals (31 each).

In [ ]:
jun10 = (spark.table("bronze_rentals")
         .withColumn("fname", F.regexp_extract("source_file", r"([^/]+\.csv)$", 1))
         .filter(F.col("fname").like("rentals_2026-06-10%")))

(jun10.groupBy("fname").count().orderBy("fname").show(truncate=False))
# -> rentals_2026-06-10.csv  31  and  rentals_2026-06-10_resend.csv  31
print("Two files carry the same 31 rentals -> this duplication is removed in Silver, not here.")